In [2]:
# %pip install faiss-cpu llama-index-vector-stores-faiss llama-index-llms-ollama llama-index-embeddings-ollama

In [7]:
import faiss
from llama_index.vector_stores.faiss import FaissVectorStore
from llama_index.core import StorageContext, load_index_from_storage

from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

In [8]:
# LLM과 임베딩 모델 설정
llm = Ollama(
    model = 'gemma2:2b',
    temperature = 0.5, # 생성되는 텍스트의 다양성을 조절하는 매개변수
    request_timeout = 120.0 # 요청이 타임아웃되기까지의 시간(초)_이 시간이 되면 멈춰라
)

embed_model = OllamaEmbedding(
    model_name = 'nomic-embed-text'
)

In [9]:
# 데이터 로드
documents = SimpleDirectoryReader('../Data/pdf_sample2/').load_data()

In [10]:
# FAISS 인덱스 생성
dimension = 768 # 임베딩 벡터의 차원 수
faiss_index = faiss.IndexFlatL2(dimension) # L2 거리 기반 인덱스 생성_ 레이어 2를 쓰는 건 X, y를 쓰는건데 _ 유사도 분석을 위해서 이걸 쓴것임

In [12]:
# 스토어 인덱스랑 벡터스토어 인덱스랑 붙여줘야 함
# FAISS를 Llamaindex의 인덱싱 및 검색 파이프라인에 통합

vector_store = FaissVectorStore(faiss_index=faiss_index) # FAISS 벡터 스토어 생성
storage_context = StorageContext.from_defaults(vector_store=vector_store) # 스토리지 컨텍스트 생성
# 연결할 준비를 한 것

In [13]:
# 인덱스 생성 및 데이터 임베딩
index = VectorStoreIndex.from_documents(
    documents, # 데이터 
    storage_context = storage_context, # 스토리지 컨텍스트
    embed_model = embed_model,
    show_progress = True # 진행 상황 표시 여부
    )

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating embeddings: 100%|██████████| 12/12 [00:02<00:00,  4.68it/s]


In [14]:
# 저장할 디렉토리
persist_dir = "./faiss_storage"

# 인덱스 저장
index.storage_context.persist(persist_dir = persist_dir)
# 원래 저장 안되는데 저장 해서 볼 예정

In [16]:
# 저장된 FAISS 벡터 스토어
loaded_vector_store = FaissVectorStore.from_persist_dir(persist_dir)

2026-05-19 11:30:33,300 - INFO - Loading llama_index.vector_stores.faiss.base from ./faiss_storage/default__vector_store.json.


In [17]:
# 저장된 인덱스 로드 위치 정의
loaded_storage_context = StorageContext.from_defaults(
    vector_store = loaded_vector_store,
    persist_dir = persist_dir)


# Llmaindex의 메터 데이터를 저장
loaded_index = load_index_from_storage(
    loaded_storage_context,
    embed_model = embed_model
)


2026-05-19 11:30:36,534 - INFO - Loading all indices.


---
### 메모리 로드

In [20]:
# 쿼리 엔진
query_engine = index.as_query_engine(llm = llm)
# 저장된거 불러서 만든 새로운 엔진

In [21]:
# 쿼리 실행
query = "이 논문에서 제인하는 모델의 장점은 무엇인가요? 한글로 답변 하세요."
response = query_engine.query(query)
# loaded_query_engine : 저장된 인덱스에서 만든 쿼리 엔진
# 응답 출력
print("\n질문: ", query)
print("답변: ", response)
# 더 자세한 답변을 원하면 파라메터 중 temperature 값을 높이면 된다 

2026-05-19 11:31:37,380 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-19 11:31:49,292 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"



질문:  이 논문에서 제인하는 모델의 장점은 무엇인가요? 한글로 답변 하세요.
답변:  이 논문에서는 Transformer 모델이 기존 모델보다 높은 BLEU 점수를 얻을 수 있고, 더 적은 학습 시간으로 동일한 성능을 보여주는 장점을 가지고 있다고 언급합니다. 특히, Transformer 모델은 WMT 2014 데이터셋에서 번역 작업에 있어서 다른 모델보다 큰 성능 향상을 보였으며, 기존의 모델들과 비교했을 때 더 적은 학습 시간으로 동일한 성능을 보여주는 장점이 있음을 알 수 있습니다. 



#### 파일에서 불러온 내용

In [22]:
# 쿼리 엔진
query_engine = loaded_index.as_query_engine(llm = llm)

In [23]:
# 쿼리 실행
query = "이 논문에서 제인하는 모델의 장점은 무엇인가요? 한글로 답변 하세요."
response = query_engine.query(query)
# 응답 출력
print("\n질문: ", query)
print("답변: ", response)
# 더 자세한 답변을 원하면 파라메터 중 temperature 값을 높이면 된다 

2026-05-19 11:32:36,089 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-19 11:32:41,298 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"



질문:  이 논문에서 제인하는 모델의 장점은 무엇인가요? 한글로 답변 하세요.
답변:  이 논문에서는 Transformer 모델이 기존 모델보다 높은 BLEU 점수를 얻을 수 있고, 더 빠른 학습 속도와 비교적 저렴한 학습비용으로 좋은 성능을 보여준다고 언급됩니다.  특히, 이 논문에서 제시된 Transformer 모델은 WMT 2014 English-to-German 및 English-to-French 번역 작업에서 높은 BLEU 점수를 달성했습니다. 또한, 다른 모델보다 학습 시간을 단축하면서 좋은 성능을 보여주는 것이 특징입니다.  

